# UI Adapters Testing

**Purpose**: Test Jupyter widget integration with the new core module

**Status**: Development notebook

**Date**: January 10, 2026

## Setup

Import required libraries and check if ipywidgets is available.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import cached_property

# Core module
from bmcs_cross_section.core import BMCSModel, ui_field

# Try to import UI adapters
try:
    from bmcs_cross_section.core.ui.jupyter import create_interactive_plot, create_widget
    JUPYTER_UI_AVAILABLE = True
    print("✅ Jupyter UI adapter available")
except ImportError as e:
    JUPYTER_UI_AVAILABLE = False
    print(f"⚠️ Jupyter UI not available: {e}")
    print("   Install with: pip install ipywidgets ipympl")

## Test Model

Create a simple model for testing UI widgets.

In [ ]:
class SimpleLinearModel(BMCSModel):
    """Simple linear function for testing"""
    
    a: float = ui_field(
        2.0,
        label=r"Slope $a$",
        unit="-",
        range=(0.0, 10.0),
        step=0.1,
        description="Slope of the line"
    )
    
    b: float = ui_field(
        3.0,
        label=r"Intercept $b$",
        unit="-",
        range=(-10.0, 10.0),
        step=0.5,
        description="Y-intercept"
    )
    
    x_min: float = ui_field(
        0.0,
        label=r"$x_{min}$",
        range=(-10.0, 10.0),
        description="Minimum x value"
    )
    
    x_max: float = ui_field(
        10.0,
        label=r"$x_{max}$",
        range=(0.0, 20.0),
        description="Maximum x value"
    )
    
    @cached_property
    def x_range(self) -> np.ndarray:
        """Get x range for plotting"""
        return np.linspace(self.x_min, self.x_max, 100)
    
    def compute_y(self, x: np.ndarray) -> np.ndarray:
        """Compute y = a*x + b"""
        return self.a * x + self.b
    
    def plot_on_axes(self, ax):
        """Plot the function on given axes"""
        x = self.x_range
        y = self.compute_y(x)
        
        ax.clear()
        ax.plot(x, y, 'b-', linewidth=2, label=f'$y = {self.a:.1f}x + {self.b:.1f}$')
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
        ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title('Linear Function')
        ax.legend()
        ax.grid(True, alpha=0.3)

# Create model instance
model = SimpleLinearModel()
print(f"Model created: {model}")

## Test 1: Basic Plot (Non-Interactive)

First, verify the plot function works without interactivity.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
model.plot_on_axes(ax)
plt.tight_layout()
plt.show()
print("✅ Basic plotting works!")

## Test 2: Interactive Plot

Create an interactive widget with sliders (requires ipywidgets).

In [ ]:
if JUPYTER_UI_AVAILABLE:
    # Create interactive plot
    # This will display sliders for a, b, x_min, x_max
    # and update the plot when you move the sliders
    create_interactive_plot(
        model,
        lambda m, ax: m.plot_on_axes(ax),
        figsize=(10, 6)
    )
    print("✅ Interactive plot created!")
else:
    print("⚠️ Skipping interactive test - ipywidgets not available")
    print("   Install with: pip install ipywidgets")
    print("   Then restart kernel and run: jupyter nbextension enable --py widgetsnbextension")

## Test 3: Widget Panel Only

Create just the widget panel without plot (for testing UI generation).

In [ ]:
if JUPYTER_UI_AVAILABLE:
    def on_change(m):
        """Callback when parameters change"""
        print(f"Parameters changed: a={m.a:.2f}, b={m.b:.2f}")
    
    # Create widget panel
    create_widget(model, on_change=on_change)
    print("✅ Widget panel created!")
else:
    print("⚠️ Skipping widget test")

## Test 4: Material Model Example

Create a more realistic example - a simple steel stress-strain model.

In [ ]:
class SimpleSteelModel(BMCSModel):
    """Bilinear steel stress-strain model"""
    
    E_s: float = ui_field(
        200000.0,
        label=r"$E_s$",
        unit="MPa",
        range=(150000.0, 250000.0),
        step=1000.0,
        description="Young's modulus",
        gt=0
    )
    
    f_y: float = ui_field(
        500.0,
        label=r"$f_y$",
        unit="MPa",
        range=(400.0, 600.0),
        step=10.0,
        description="Yield strength",
        gt=0
    )
    
    eps_max: float = ui_field(
        0.01,
        label=r"$\varepsilon_{max}$",
        unit="-",
        range=(0.005, 0.02),
        step=0.001,
        description="Maximum strain for plot"
    )
    
    @cached_property
    def eps_y(self) -> float:
        """Yield strain"""
        return self.f_y / self.E_s
    
    def get_sig(self, eps: np.ndarray) -> np.ndarray:
        """
        Compute stress from strain (bilinear).
        
        Args:
            eps: Strain array
            
        Returns:
            Stress array [MPa]
        """
        eps_y = self.eps_y
        sig = np.where(
            np.abs(eps) <= eps_y,
            self.E_s * eps,  # Elastic
            np.sign(eps) * self.f_y  # Plastic
        )
        return sig
    
    def plot_stress_strain(self, ax):
        """Plot stress-strain curve"""
        eps = np.linspace(-self.eps_max, self.eps_max, 200)
        sig = self.get_sig(eps)
        
        ax.clear()
        ax.plot(eps, sig, 'b-', linewidth=2)
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        
        # Mark yield points
        ax.plot([-self.eps_y, self.eps_y], [-self.f_y, self.f_y], 
                'ro', markersize=8, label=f'Yield: ε={self.eps_y:.4f}')
        
        ax.set_xlabel('Strain ε [-]')
        ax.set_ylabel('Stress σ [MPa]')
        ax.set_title('Bilinear Steel Stress-Strain')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-self.eps_max, self.eps_max)

# Create steel model
steel = SimpleSteelModel()
print(f"Steel model: E_s={steel.E_s} MPa, f_y={steel.f_y} MPa")
print(f"Yield strain: {steel.eps_y:.4f}")

## Test 5: Interactive Steel Model

Create interactive widget for the steel model.

In [ ]:
if JUPYTER_UI_AVAILABLE:
    # Create interactive stress-strain plot
    create_interactive_plot(
        steel,
        lambda m, ax: m.plot_stress_strain(ax),
        figsize=(10, 6)
    )
    print("✅ Interactive steel model created!")
    print("   Try moving the sliders to see real-time updates!")
else:
    # Fallback: static plot
    fig, ax = plt.subplots(figsize=(10, 6))
    steel.plot_stress_strain(ax)
    plt.tight_layout()
    plt.show()
    print("⚠️ Static plot shown (install ipywidgets for interactive version)")

## Test 6: Efficient Update Mode

Demonstrate the efficient update pattern that only changes data, not the entire figure.

In [ ]:
if JUPYTER_UI_AVAILABLE:
    # Define setup function (called once)
    def setup_steel_plot(model, ax):
        """Setup axes and create plot artists (called once)"""
        ax.set_xlabel('Strain ε [-]')
        ax.set_ylabel('Stress σ [MPa]')
        ax.set_title('Bilinear Steel Stress-Strain (Efficient Update)')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        
        # Create line and point artists
        line, = ax.plot([], [], 'b-', linewidth=2, label='σ-ε curve')
        yield_points, = ax.plot([], [], 'ro', markersize=8, label='Yield points')
        ax.legend()
        
        # Return artists for updating
        return {'line': line, 'yield_points': yield_points}
    
    # Define update function (called on each parameter change)
    def update_steel_plot(model, ax, artists):
        """Update only the plot data (efficient)"""
        # Compute new data
        eps = np.linspace(-model.eps_max, model.eps_max, 200)
        sig = model.get_sig(eps)
        
        # Update line data
        artists['line'].set_data(eps, sig)
        
        # Update yield points
        artists['yield_points'].set_data(
            [-model.eps_y, model.eps_y], 
            [-model.f_y, model.f_y]
        )
        
        # Update axis limits if needed
        ax.set_xlim(-model.eps_max, model.eps_max)
        ax.relim()
        ax.autoscale_view(scalex=False)  # Keep x limits, auto-scale y
    
    # Create interactive plot with efficient updates
    create_interactive_plot(
        steel,
        setup_steel_plot,  # Setup function
        update_steel_plot,  # Update function
        figsize=(10, 6)
    )
    print("✅ Efficient interactive plot created!")
    print("   Notice: No flickering - only the curve updates!")
else:
    print("⚠️ Skipping efficient update test")

## Summary

### What We Tested:
1. ✅ Basic model with UI metadata
2. ✅ Static plotting
3. ✅ Interactive widgets (if ipywidgets available)
4. ✅ Parameter callbacks
5. ✅ Realistic material model example

### UI Features Working:
- Automatic slider generation from `ui_field()`
- LaTeX labels rendering
- Units displayed
- Range constraints enforced
- Real-time plot updates
- Cache invalidation on parameter change

### Next Steps:
- Test Streamlit adapter (separate app file)
- Create EC2 concrete model with symbolic derivation
- Validate against legacy implementation

## Notes for Streamlit

To test the Streamlit adapter, create a separate Python file:

```python
# streamlit_test.py
from bmcs_cross_section.core.ui.streamlit import create_streamlit_app
import matplotlib.pyplot as plt

def plot_func(model, fig, ax):
    model.plot_stress_strain(ax)

AppClass = create_streamlit_app(
    SimpleSteelModel,
    "Steel Material Model",
    plot_func,
    description="Interactive bilinear steel model"
)

if __name__ == "__main__":
    app = AppClass()
    app.run()
```

Then run: `streamlit run streamlit_test.py`